Dependências
- Instala o kagglehub, caso necessário
- Carrega bibliotecas necessárias para execução

In [ ]:
%pip install scikit-learn pandas matplotlib scipy kagglehub torch torchvision

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import torch
import os
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
import torch.nn as nn
import copy
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"

Carregar Dataset
- Primeiramente, baixa o dataset na maquina pelo kagglehub

In [ ]:
file_path = "HAM10000_metadata.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "kmader/skin-cancer-mnist-ham10000",
  file_path,
)
root = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print(df.head())

Criação de Classe para facilitar transformações no dataset

In [ ]:
# Tratamento Dataset
labels = dict()

for i, l in  enumerate(df['dx'].unique()):
    labels[l] = i

class HAMDataset(Dataset):
    def __init__(self, dataframe, root, transform=None):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.root = root
        self.transform = transform

        self.img_paths = []
        self.img_labels = []

        for i in range(len(self.dataframe)):
            row = self.dataframe.iloc[i]
            img_name = row['image_id'] + '.jpg'
            diagnosis = row['dx']

            # Verifica se esta na part_1 ou part_2
            img_path = os.path.join(root, 'HAM10000_images_part_1', img_name)
            if not os.path.exists(img_path):
                img_path = os.path.join(root, 'HAM10000_images_part_2', img_name)

            self.img_paths.append(img_path)
            self.img_labels.append(labels[diagnosis])

        print(f"Dataset preparado: {len(self.img_paths)} imagens válidas de {len(dataframe)} totais")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.item()

        img_path = self.img_paths[idx]
        label = self.img_labels[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label


Criação das transformações a serem aplicadas nas imagens (Vetorização, Redimensionalização, Normalização...)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['dx'],
    random_state=42
)

print(f"Treino: {len(train_df)} imagens | Teste: {len(test_df)} imagens")
print(f"Distribuição de classes no treino:\n{train_df['dx'].value_counts()}\n")

train_dataset = HAMDataset(dataframe=train_df, root=root, transform=train_transform)
test_dataset  = HAMDataset(dataframe=test_df,  root=root, transform=test_transform)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


In [ ]:
# Função treino generalizada para todos os modelos
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(dataloader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if batch_idx % 50 == 0:
            print(f'Batch {batch_idx}/{len(dataloader)} - Loss: {loss.item():.2f}')

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

Aplicação de vários modelos de treino, para avaliação do melhor.
Em conjunto, seleção dos parametros para cada determinado modelo.

In [ ]:
# Configurações Especiais pra teste
# Lista de modelos para testar
lst_models_config = [
    # Modelos classicos
    {'name': 'VGG16', 'class': models.vgg16, 'pretrained': True},
    {'name': 'ResNet18', 'class': models.resnet18, 'pretrained': True},
    {'name': 'ResNet50', 'class': models.resnet50, 'pretrained': True},

    # EfficientNets
    {'name': 'EfficientNet-B0', 'class': models.efficientnet_b0, 'pretrained': True},
    {'name': 'EfficientNet-B2', 'class': models.efficientnet_b2, 'pretrained': True},

    # Transformers
    {'name': 'SwinTransformer', 'class': models.swin_t, 'pretrained': True},

    # Balanceados
    {'name': 'DenseNet121', 'class': models.densenet121, 'pretrained': True},
    {'name': 'MobileNetV3', 'class': models.mobilenet_v3_large, 'pretrained': True},

    # Especificos
    {'name': 'ConvNeXt', 'class': models.convnext_tiny, 'pretrained': True}
]

lst_num_epochs = [3, 5, 10]

def create_model(model_config, num_classes=7, freeze_backbone=True):
    model_name = model_config['name']
    model_class = model_config['class']
    pretrained = model_config['pretrained']

    # Carregar modelo
    model = model_class(pretrained=pretrained)

    # Congelar backbone se necessário
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Modificar última camada para 7 classes
    if 'vgg' in model_name.lower():
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            # Descongelar apenas classifier
            for param in model.classifier.parameters():
                param.requires_grad = True

    elif 'resnet' in model_name.lower():
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.fc.parameters():
                param.requires_grad = True

    elif 'efficientnet' in model_name.lower():
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.classifier.parameters():
                param.requires_grad = True

    elif 'vit' in model_name.lower():
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.heads.parameters():
                param.requires_grad = True

    elif 'swin' in model_name.lower():
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.head.parameters():
                param.requires_grad = True

    elif 'densenet' in model_name.lower():
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.classifier.parameters():
                param.requires_grad = True

    elif 'mobilenet' in model_name.lower():
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.classifier.parameters():
                param.requires_grad = True

    elif 'inception' in model_name.lower():
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)
        if freeze_backbone:
            for param in model.fc.parameters():
                param.requires_grad = True
            for param in model.AuxLogits.parameters():
                param.requires_grad = True

    elif 'convnext' in model_name.lower():
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
        if freeze_backbone:
            for param in model.classifier.parameters():
                param.requires_grad = True

    else:
        raise ValueError(f"Modelo {model_name} não suportado")

    return model, model_name

In [ ]:
# Criar diretório para salvar modelos
os.makedirs("models", exist_ok=True)

Execução dos treinos e avaliações dos modelos

In [ ]:
# Lista para armazenar todos os resultados
all_results = []

# Treinamento
for model_config in lst_models_config:
    model_name = model_config['name']
    print(f"Modelo: {model_name}")

    for num_epochs in lst_num_epochs:
        print(f"Treinando {model_name} por {num_epochs} epochs")

        # Novo modelo para experimento
        model, _ = create_model(model_config, num_classes=7, freeze_backbone=True)
        model = model.to(device)

        criterion = nn.CrossEntropyLoss()
        trainable_params = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = optim.Adam(trainable_params, lr=0.001)

        train_losses = []
        train_accuracies = []

        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            loss, acc = train_one_epoch(model, train_dataloader, criterion, optimizer, device)
            train_losses.append(loss)
            train_accuracies.append(acc)
            print(f"\tLoss: {loss:.4f}, Accuracy: {acc:.2%}")

        # Avaliação
        print("Avaliação no conjunto de teste")

        model.eval()
        test_correct = 0
        test_total = 0
        all_labels = []
        all_predictions = []

        with torch.no_grad():
            for images, labels in test_dataloader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())

        test_accuracy = test_correct / test_total

        print(f"\nAccuracy no teste: {test_accuracy:.2%}")
        print(f"Corretos: {test_correct}/{test_total}")

        cm = confusion_matrix(all_labels, all_predictions)
        print(f"\nMatriz de confusão (diagonal): {cm.diagonal()}")

        print("\nRelatório de classificação:")
        print(classification_report(all_labels, all_predictions,
                                   target_names=sorted(df['dx'].unique()),
                                   digits=4))

        results = {
            'model': model_name,
            'epochs': num_epochs,
            'test_accuracy': test_accuracy,
            'train_loss_final': train_losses[-1],
            'train_acc_final': train_accuracies[-1],
            'test_correct': test_correct,
            'test_total': test_total,
            'confusion_matrix': cm.tolist()
        }

        all_results.append(results)

        # Salvar Modelo
        save_dir = "models"
        os.makedirs(save_dir, exist_ok=True)
        save_path = f"{save_dir}/{model_name.replace(' ', '_')}_{num_epochs}epochs.pth"

        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epochs': num_epochs,
            'accuracy': test_accuracy,
            'results': results
        }, save_path)
        print(f"\nModelo salvo em: {save_path}")

        print(f"Fim: {model_name} - {num_epochs} epochs")


Resultados obtidos em uma das execuções:

Resumo Todos Experimentos

1. VGG16 - 3 epochs
        Accuracy: 69.30%
        Loss final: 1.0018

2. VGG16 - 5 epochs
        Accuracy: 72.14%
        Loss final: 0.9645

3. VGG16 - 10 epochs
        Accuracy: 73.69%
        Loss final: 0.8748

4. ResNet18 - 3 epochs
        Accuracy: 72.74%
        Loss final: 0.7054

5. ResNet18 - 5 epochs
        Accuracy: 74.04%
        Loss final: 0.6700

6. ResNet18 - 10 epochs
        Accuracy: 76.64%
        Loss final: 0.6349

7. ResNet50 - 3 epochs
        Accuracy: 74.74%
        Loss final: 0.7054

8. ResNet50 - 5 epochs
        Accuracy: 76.39%
        Loss final: 0.6764

9. ResNet50 - 10 epochs
        Accuracy: 76.78%
        Loss final: 0.6526

10. EfficientNet-B0 - 3 epochs
        Accuracy: 75.39%
        Loss final: 0.7043

11. EfficientNet-B0 - 5 epochs
        Accuracy: 75.99%
        Loss final: 0.6730

12. EfficientNet-B0 - 10 epochs
        Accuracy: 77.28%
        Loss final: 0.6286

13. EfficientNet-B2 - 3 epochs
        Accuracy: 74.89%
        Loss final: 0.7020

14. EfficientNet-B2 - 5 epochs
        Accuracy: 76.54%
        Loss final: 0.6723

15. EfficientNet-B2 - 10 epochs
        Accuracy: 76.59%
        Loss final: 0.6352

16. SwinTransformer - 3 epochs
        Accuracy: 74.69%
        Loss final: 0.7003

17. SwinTransformer - 5 epochs
        Accuracy: 77.08%
        Loss final: 0.6456

18. SwinTransformer - 10 epochs
        Accuracy: 77.83%
        Loss final: 0.5987

19. DenseNet121 - 3 epochs
        Accuracy: 76.49%
        Loss final: 0.6700

20. DenseNet121 - 5 epochs
        Accuracy: 77.53%
        Loss final: 0.6327

21. DenseNet121 - 10 epochs
        Accuracy: 78.13%
        Loss final: 0.5863

22. MobileNetV3 - 3 epochs
        Accuracy: 77.98%
        Loss final: 0.5942

23. MobileNetV3 - 5 epochs
        Accuracy: 78.58%
        Loss final: 0.5185

24. MobileNetV3 - 10 epochs
        Accuracy: 77.58%
        Loss final: 0.3954

25. ConvNeXt - 3 epochs
        Accuracy: 77.03%
        Loss final: 0.6394

26. ConvNeXt - 5 epochs
        Accuracy: 78.98%
        Loss final: 0.5923

27. ConvNeXt - 10 epochs
        Accuracy: 80.03%
        Loss final: 0.5286
Melhor Modelo: ConvNeXt com 10 epochs
Acuracia: 80.03%

Resumo dos testes

In [ ]:
print("Resumo Todos Experimentos")

for i, result in enumerate(all_results):
    print(f"\n{i+1}. {result['model']} - {result['epochs']} epochs")
    print(f"\tAccuracy: {result['test_accuracy']:.2%}")
    print(f"\tLoss final: {result['train_loss_final']:.4f}")

if all_results:
    best_result = max(all_results, key=lambda x: x['test_accuracy'])
    print(f"Melhor Modelo: {best_result['model']} com {best_result['epochs']} epochs")
    print(f"Acuracia: {best_result['test_accuracy']:.2%}")

Treino executado apenas na ConvNeXt, 50 epochs, early stop:

In [ ]:
train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df['dx'],
    random_state=42
)

# Depois, divide para validação
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.15/0.85,  # Para manter proporção correta
    stratify=train_val_df['dx'],
    random_state=42
)

print(f"Treino: {len(train_df)} imagens")
print(f"Validação: {len(val_df)} imagens")
print(f"Teste: {len(test_df)} imagens")

# Criar datasets
train_dataset = HAMDataset(dataframe=train_df, root=root, transform=train_transform)
val_dataset = HAMDataset(dataframe=val_df, root=root, transform=test_transform)
test_dataset = HAMDataset(dataframe=test_df, root=root, transform=test_transform)

# Criar dataloaders
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
def train_with_early_stopping(model, train_loader, val_loader, criterion, optimizer, device, 
                               max_epochs=50, patience=5, min_delta=0.001):
    model.to(device)
    best_val_acc = 0.0
    best_model_state = None
    epochs_no_improve = 0
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            if batch_idx % 50 == 0:
                print(f'  Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}')
        
        train_loss = train_loss / len(train_loader)
        train_acc = train_correct / train_total
        
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = val_correct / val_total
        
        # Salvar historico
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        # Mostrar resultados
        print(f'\nEpoch {epoch+1}/{max_epochs}:')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2%}')
        print(f'  Val Acc: {val_acc:.2%}')
        
        if val_acc > best_val_acc + min_delta:
            best_val_acc = val_acc
            epochs_no_improve = 0
            best_model_state = model.state_dict().copy()
            print(f'Melhor modelo, Val Acc: {best_val_acc:.2%}')
        else:
            epochs_no_improve += 1
            
            if epochs_no_improve >= patience:
                print(f'\nEarly Stopping ativado na epoch {epoch+1}')
                break
    
    # Carregar melhor modelo
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f'\nMelhor modelo carregado (Val Acc: {best_val_acc:.2%})')
    
    return model, history, best_val_acc

In [ ]:
model_config = {'name': 'ConvNeXt', 'class': models.convnext_tiny, 'pretrained': True}
model, model_name = create_model(model_config, num_classes=7, freeze_backbone=True)
criterion = nn.CrossEntropyLoss()
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(trainable_params, lr=0.001)

In [ ]:
print("Treinando modelo: ConvNeXt")

model, history, best_val_acc = train_with_early_stopping(
    model=model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    max_epochs=50,
    patience=5,
    min_delta=0.001
)

In [ ]:
# Avaliação final no teste
model.eval()
test_correct = 0
test_total = 0
all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

test_accuracy = test_correct / test_total

print(f"\nAccuracy no teste: {test_accuracy:.2%}")
print(f"Corretos: {test_correct}/{test_total}")

# Matriz de confusão
cm = confusion_matrix(all_labels, all_predictions)
print(f"\nMatriz de confusão (diagonal): {cm.diagonal()}")

# Relatório de classificação
class_names = sorted(labels.keys())
print("\nRelatório de classificação:")
print(classification_report(all_labels, all_predictions,
                           target_names=class_names,
                           digits=4))

# Salvar resultados
results = {
    'model': model_name,
    'test_accuracy': test_accuracy,
    'val_accuracy': best_val_acc,
    'test_correct': test_correct,
    'test_total': test_total,
    'confusion_matrix': cm.tolist(),
    'history': history
}

# Salvar modelo
save_dir = "models"
os.makedirs(save_dir, exist_ok=True)
save_path = f"{save_dir}/{model_name.replace(' ', '_')}_final.pth"

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'accuracy': test_accuracy,
    'results': results
}, save_path)
print(f"\nModelo salvo em: {save_path}")

print("TREINAMENTO CONCLUÍDO!")

Accuracy no teste: 80.97%
Corretos: 1217/1503

Relatório de classificação:
              precision    recall  f1-score   support

       akiec     0.7244    0.5576    0.6301       165
         bcc     0.8484    0.9682    0.9044      1006
         bkl     0.7692    0.5882    0.6667        17
          df     0.6935    0.2575    0.3755       167
         mel     1.0000    0.5909    0.7429        22
          nv     0.7077    0.5974    0.6479        77
        vasc     0.5200    0.7959    0.6290        49

    accuracy                         0.8097      1503
   macro avg     0.7519    0.6222    0.6566      1503
weighted avg     0.8010    0.8097    0.7883      1503

# Respostas Slides

1. De acordo com os resultados obtidos, a rede escolhida para avaliação foi:
-  ConvNeXt - 10 epochs, com uma acurária de aprox. **80**%, devido a sua melhor acurácia no periodo de tempo semelhante aos outros.

2. As adaptações realizadas, para testes e comparação de resultados, foram principalmente:
- Aplicação de dataloaders, para carregamento em batches dos dados.
- Transformações nas imagens para encaixar melhor nos modelos (Redimensionalização, Normalização, Vetorização...)
- Aplicação de vários modelos de redes diferentes para comparação (com parametros diferentes para cada rede).
- Variação na quantidade de epochs no treinamento (3, 5 e 10, quantidades para o colab suportar execução de todas as redes).

3. Ao executar apenas o modelo, com no máximo 50 epochs e early stop, a acurácia máxima obtida foi de aprox. **82**%, não se afastando muito do resultado obtido anteriormente, mas executando em 33 epochs (aprox. 20 epochs a mais por 2% de acurácia). No ponto de vista de tradeoff acurácia x tempo de execução, não há ganho signficativo ao rodar mais epochs para o modelo nesse determinado dataset.

